# Case 1 — Clean-data false-positive baseline

**Reproduces:** Fig 4.1 (score histograms), Fig 4.2 (rolling FPR), Table 4.1

No anomalies are injected here — every predicted 'anomaly' is a false positive. The thesis found Transformer-VAE fits the warmup data so tightly that its rolling threshold starts miscalibrated and produces an early false-positive burst before adapting, while MLP-VAE-Cyclic stays calibrated from the start. On the accuracy column below, lower is better — it is literally the false-positive rate.

Runtime menu -> Change runtime type -> GPU, then run all cells.

This notebook runs on the small synthetic ERA5-shaped dataset shipped with the repo (`scripts/generate_mini_era5.py`) — no data download, no license issues. Numbers will differ from the thesis's real-ERA5 figures (much smaller warmup/test period, noisier), but the *qualitative* effect described above should still show up.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# TODO: update this URL once the repo is pushed to GitHub
REPO_URL = "https://github.com/<your-username>/streaming-vae-anomaly-detection.git"

!git clone $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# Generates data/era5/*.csv — fully synthetic, ERA5-shaped, no download needed
!python scripts/generate_mini_era5.py

## Run the suite

`notebooks/cases/case01_clean_baseline_suite.yaml` layers `modules/era5_common.yaml` with the architecture / anomaly-type / stream-mode / feature-engineering fragments for each of the runs below (see the suite file for the exact overrides). Runs execute sequentially on the one Colab GPU; each is small (mini dataset), so the whole case should finish in a few minutes.

In [ ]:
!python run_regression.py notebooks/cases/case01_clean_baseline_suite.yaml \
    --session runs/regression/case01_clean_baseline --gpus 0

## Compare the runs

`cross_compare.py` walks the session directory, extracts F1/AUC/precision/recall from each run's `trial_predictions.csv`, and writes a performance table plus comparison plots (score histograms, F1-vs-variant line plot, confusion grid, seed-stability heatmap) under `<session>/cross_compare/`.

In [ ]:
!python cross_compare.py runs/regression/case01_clean_baseline

In [ ]:
import pandas as pd
perf = pd.read_csv("runs/regression/case01_clean_baseline/cross_compare/performance_table.csv")
perf[["run_name", "arch", "anomaly", "variant", "f1", "auc", "precision", "recall", "tp", "fp", "fn"]]

## Read the false-positive rate directly

`accuracy` in the table above is `1 - FPR` here since every sample is labelled normal (no positives exist, so F1/precision/recall are undefined — `fp` and `tp+fp+tn+fn` are what matters). Lower `fp` = better calibration.

In [ ]:
perf['fpr_pct'] = 100 * perf['fp'] / (perf['fp'] + perf['tn'])
perf[['run_name', 'arch', 'fp', 'tn', 'fpr_pct']]